In [8]:
%%capture

# Install dependencies specified in pyproject.toml
!uv pip install -r pyproject.toml

from pathlib import Path

# Import modules
from datetime import datetime
import cma
from IPython.display import display
import lumapi as lma
import math
import matplotlib.pyplot as plt
import numpy as np
import photonix as px

In [9]:
# Constants

um = 1e-6
nm = 1e-9
C = 299792458

In [10]:
# Si3N4/SiO2 1000x360nm 1550nm
experiment_name = ""
wavelength_center = 1.55*um
wavelength_span = 0.15*um

dc_sparams = "results/si3n4_w1000nm_h360nm_directCoupler_g400nm_l12um_man.dat"


In [11]:
calc_freq = lambda wvl: (C/wvl)
calc_mw = lambda dbm: 10.0 ** (dbm / 10.0)

def remove(object_name):
    try:
        solver.select(object_name)
        solver.delete()
    except:
        pass

In [12]:
script_cwd = Path.cwd()

dc_sparams_abs = f"{script_cwd}/{dc_sparams}"

In [13]:
wavelength_min = wavelength_center - wavelength_span/2
wavelength_max = wavelength_center + wavelength_span/2

frequency_center = calc_freq(wavelength_center)
frequency_min = calc_freq(wavelength_max)
frequency_max = calc_freq(wavelength_min)
frequency_span = frequency_max - frequency_min

ic_cell_size = 175

In [14]:
# Get solver
solver = lma.INTERCONNECT()

LumApiError: "Could not open 'interconnect': appOpen error: Application '/opt/lumerical/current/bin/interconnect' did not produce the startup UUID within 60 s (exit code 255).\nstderr:\nlibGL error: MESA-LOADER: failed to open radeonsi: /opt/lumerical/current/bin/../lib/mesa/dri/radeonsi_dri.so: cannot open shared object file: No such file or directory (search paths /opt/lumerical/current/bin/../lib/mesa/dri, suffix _dri)\nlibGL error: failed to load driver: radeonsi\nWeb Licensing checkout call to Direct LaaS REST API for 1 tasks of lum_interconnect_gui failed:\n\nError Detail: Curl error 6 encountered on call to https://laas.ansys.com/v1/products/checkin?format=signed&verbose=true: Could not resolve host: laas.ansys.com\n\nStatus code: -6"

In [103]:
solver.switchtodesign()

# Optical Network Analyzer
remove("ONA-1")
solver.addelement(
    "Optical Network Analyzer",
    name="ONA-1",
    input_parameter="center and range",
    number_of_input_ports=2,
    center_frequency=frequency_center,
    frequency_range=frequency_span,
    x_position=0,
    y_position=0,
)

# Directional Coupler
remove("DC-1")
solver.addelement(
    "Waveguide coupler",
    name="DC-1",
    x_position=0,
    y_position=200,
)
# solver.addelement(
#     "Optical N Port S-Parameter",
#     name="DC-1",
#     load_from_file=True,
#     s_parameters_filename=dc_sparams_abs,
#     x_position=0,
#     y_position=200,
# )

solver.connect("ONA-1", "output", "DC-1", "port 1")
solver.connect("DC-1", "port 3", "ONA-1", "input 1")
solver.connect("DC-1", "port 4", "ONA-1", "input 2")


1.0

In [138]:
# Gate

# Optical Network Analyzer
remove("ONA-1")
solver.addelement(
    "Optical Network Analyzer",
    name="ONA-1",
    input_parameter="center and range",
    number_of_input_ports=2,
    center_frequency=frequency_center,
    frequency_range=frequency_span,
    x_position=0,
    y_position=-1.5*ic_cell_size,
)

# Directional Coupler
remove("DC-1")
solver.addelement(
    "Waveguide coupler",
    name="DC-1",
    x_position=-1*ic_cell_size,
    y_position=0,
)

# Phase shifter 1
remove("PS-1")
solver.addelement(
    "Optical Phase Shift",
    name="PS-1",
    x_position=0,
    y_position=-0.5*ic_cell_size,
)

# Phase shifter 2
remove("PS-2")
solver.addelement(
    "Optical Phase Shift",
    name="PS-2",
    x_position=0,
    y_position=0.5*ic_cell_size,
)

# Directional Coupler
remove("DC-2")
solver.addelement(
    "Waveguide coupler",
    name="DC-2",
    x_position=1*ic_cell_size,
    y_position=0,
)

solver.connect("ONA-1", "output", "DC-1", "port 1")
solver.connect("DC-1", "port 3", "PS-1", "port 1")
solver.connect("DC-1", "port 4", "PS-2", "port 1")
solver.connect("PS-1", "port 2", "DC-2", "port 1")
solver.connect("PS-2", "port 2", "DC-2", "port 2")
solver.connect("DC-2", "port 3", "ONA-1", "input 1")
solver.connect("DC-2", "port 4", "ONA-1", "input 2")



# # Phase shifter 1
# remove("PS-2")
# solver.addelement(
#     "Optical Phase Shift",
#     name="P2-1",
#     x_position=1*ic_cell_size,
#     y_position=1*ic_cell_size,
# )

1.0

In [220]:
# Gate (2 inputs)

def set_input(A=0, B=0):
    power = 0.01  # 10 mW
    solver.switchtodesign()
    solver.setnamed("LS-1", "power", float(A * power))
    solver.setnamed("LS-2", "power", float(B * power))

def get_output():
    o1 = solver.getresult("PM-1", "sum/power")
    o2 = solver.getresult("PM-2", "sum/power")

    return (o1, o2)

def set_phase(A=0, B=0):
    solver.switchtodesign()
    solver.setnamed("PS-1", "phase shift", math.radians(A))
    solver.setnamed("PS-2", "phase shift", math.radians(B))
    


def run_sim():
    laser_power_mw = 10
    cases = [(0, 0), (0, 1), (1, 0), (1, 1)]

    header = f"{'A (mW)':>6} | {'B (mW)':>6} | {'O1 (mW)':>6} | {'O1 (%)':>6}"
    print(header)
    print("-" * len(header))

    for a, b in cases:
        set_input(A=a, B=b)
        solver.run()
        o1, o2 = get_output()

        (a_mw, b_mw) = a * laser_power_mw, b * laser_power_mw
        (o1_mw, o2_mw) = calc_mw(o1), calc_mw(o2)
        
        print(f"{a_mw:6.1f} | {b_mw:6.1f} | {o1_mw:6.1f} | {o1_mw/laser_power_mw:6.1f}")
    
solver.switchtodesign()

# Laser1
remove("LS-1")
solver.addelement(
    "Multimode CW Laser",
    name="LS-1",
    frequency=frequency_center,
    x_position=-1*ic_cell_size,
    y_position=-0.5*ic_cell_size,
    power=0.01,
)

# Phase shifter 1
remove("PS-1")
solver.addelement(
    "Optical Phase Shift",
    name="PS-1",
    x_position=0,
    y_position=-0.5*ic_cell_size,
)

# Laser2
remove("LS-2")
solver.addelement(
    "Multimode CW Laser",
    name="LS-2",
    frequency=frequency_center,
    x_position=-1*ic_cell_size,
    y_position=0.5*ic_cell_size,
    power=0.01,
)

# Phase shifter 2
remove("PS-2")
solver.addelement(
    "Optical Phase Shift",
    name="PS-2",
    x_position=0,
    y_position=0.5*ic_cell_size,
)

# Directional Coupler
remove("DC-2")
solver.addelement(
    "Waveguide coupler",
    name="DC-2",
    x_position=1*ic_cell_size,
    y_position=0,
)

# Power Meter 1
remove("PM-1")
solver.addelement(
    "Optical Power Meter",
    name="PM-1",
    x_position=2*ic_cell_size,
    y_position=-0.5*ic_cell_size,
)

# Power Meter 2
remove("PM-2")
solver.addelement(
    "Optical Power Meter",
    name="PM-2",
    x_position=2*ic_cell_size,
    y_position=0.5*ic_cell_size,
)

solver.connect("LS-1", "output", "PS-1", "port 1")
solver.connect("PS-1", "port 2", "DC-2", "port 1")
solver.connect("DC-2", "port 3", "PM-1", "input")

solver.connect("LS-2", "output", "PS-2", "port 1")
solver.connect("PS-2", "port 2", "DC-2", "port 2")
solver.connect("DC-2", "port 4", "PM-2", "input")


# OR
# set_phase(A=0, B=30)
# A (mW) | B (mW) | O1 (mW) | O1 (%)
# ----------------------------------
#   0.0 |    0.0 |    0.0 |    0.0
#   0.0 |   10.0 |    5.0 |    0.5
#  10.0 |    0.0 |    5.0 |    0.5
#  10.0 |   10.0 |    5.0 |    0.5

# run_sim()

# AND
set_phase(A=0, B=0)
run_sim()


A (mW) | B (mW) | O1 (mW) | O1 (%)
----------------------------------


LumApiError: 'in run: The simulation failed to run.'